# Experiment 44 — Diverse local-credit screen on ML-1M

Short-horizon tournament over several **different families of local learning rules** on the same SparseWalker initialization and ML-1M protocol. This is an exploration experiment, not a tuned benchmark.

Arms:
- `lc_sigmoid` — current Experiment-43 analytic local contrastive rule
- `ff_npair` — N-pair / Distance-Forward-style metric local objective
- `target_route_prop` — target-propagation-inspired next-concept teaching
- `target_broadcast` — NoProp-inspired direct next-item vector broadcast
- `forward_gradient` — forward finite-difference hidden teaching signal

All use the same corrected SparseWalker v1.1, same random seed, same data split, and no end-to-end backpropagation. An e-prop-like trace arm is intentionally held out of this first screen until its persistent per-user trace implementation is validated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, torch, importlib
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
for p in [f'{REPO}/src', f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
importlib.invalidate_caches()
import sparsewalker
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('TORCH',torch.__version__,flush=True)
print('HEAD',HEAD,flush=True)
print('IMPORT_OK',sparsewalker.__file__,flush=True)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run the tournament

Default is **4 epochs per arm**. The notebook prints one compact line per epoch and a final leaderboard.


In [ ]:
import runpy, sys, os
EPOCHS_PER_ARM=4
ARMS=['lc_sigmoid','ff_npair','target_route_prop','target_broadcast','forward_gradient']
SCRIPT=f'{REPO}/experiments/run_ml1m_local_rule_screen.py'
text=Path(SCRIPT).read_text()
assert 'Experiment 44' in text and 'LOCAL_RULE_SCREEN_RESULT' in text, 'stale script clone'
print('SCRIPT_OK',SCRIPT,flush=True)
argv=[SCRIPT,
      '--epochs-per-arm',str(EPOCHS_PER_ARM),
      '--batch-size','512',
      '--eval-batch-size','1024',
      '--progress-every','0',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data',
      '--arms',*ARMS]
print('RUNNING_SCREEN',' '.join(argv),flush=True)
old_argv=sys.argv[:]; old_cwd=os.getcwd(); sys.argv=argv; os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv; os.chdir(old_cwd)


## Compare the learning-rule families


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_rule_screen_ml1m/seed42')
rp=root/'result.json'
if not rp.exists():
    print('No final result yet. Partial results:', (root/'partial_results.json').exists())
else:
    result=json.loads(rp.read_text())
    board=pd.DataFrame(result['leaderboard'])
    display(board)
    rows=[]
    for r in result['results']:
        for h in r.get('history',[]):
            rows.append({'rule':r['rule'], **h})
    hist=pd.DataFrame(rows)
    if len(hist):
        display(hist[['rule','epoch','val_NDCG@10','val_HR@10','val_MRR@10','positions_per_s','mean_signal_norm']])


## How to read the outcome

The point is not whether a 4-epoch arm beats a tuned backprop model. We want to know which *credit-assignment family* produces the strongest immediate trajectory. A clear winner gets a proper long run and tuning pass next.
